In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, roc_auc_score, precision_score,
    recall_score, f1_score, matthews_corrcoef
)
from sklearn.model_selection import cross_val_score

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import RandomForestClassifier

import xgboost as xgb
import pickle



ModuleNotFoundError: No module named 'xgboost'

In [ ]:
df = pd.read_csv("C:/Users/SESA804787/OneDrive - Schneider Electric/SE/WILP/Machine Learning/Assignment 2/heart.csv")
print("Dataset shape:", df.shape)
print("\nFirst few rows:")
df.head()


Dataset shape: (918, 12)

First few rows:


,Age,Sex,ChestPainType,RestingBP,Cholesterol,FastingBS,RestingECG,MaxHR,ExerciseAngina,Oldpeak,ST_Slope,HeartDisease
0,40,M,ATA,140,289,0,Normal,172,N,0.0,Up,0
1,49,F,NAP,160,180,0,Normal,156,N,1.0,Flat,1
2,37,M,ATA,130,283,0,ST,98,N,0.0,Up,0
3,48,F,ASY,138,214,0,Normal,108,Y,1.5,Flat,1
4,54,M,NAP,150,195,0,Normal,122,N,0.0,Up,0


In [ ]:
print("\n" + "="*50)
print("DATA EXPLORATION")
print("="*50)

print("\nDataset Info:")
print(df.info())

print("\nMissing Values:")
print(df.isnull().sum())

print("\nClass Distribution:")
print(df['HeartDisease'].value_counts())
print(f"Class balance: {df['HeartDisease'].value_counts(normalize=True)}")

print("\nNumeric Feature Summary:")
print(df.describe())




DATA EXPLORATION

Dataset Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 918 entries, 0 to 917
Data columns (total 12 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Age             918 non-null    int64  
 1   Sex             918 non-null    object 
 2   ChestPainType   918 non-null    object 
 3   RestingBP       918 non-null    int64  
 4   Cholesterol     918 non-null    int64  
 5   FastingBS       918 non-null    int64  
 6   RestingECG      918 non-null    object 
 7   MaxHR           918 non-null    int64  
 8   ExerciseAngina  918 non-null    object 
 9   Oldpeak         918 non-null    float64
 10  ST_Slope        918 non-null    object 
 11  HeartDisease    918 non-null    int64  
dtypes: float64(1), int64(6), object(5)
memory usage: 86.2+ KB
None

Missing Values:
Age               0
Sex               0
ChestPainType     0
RestingBP         0
Cholesterol       0
FastingBS         0
RestingECG        0
MaxHR  

In [ ]:
# =============================================================================
# 3. PREPROCESSING
# =============================================================================
# Separate features and target
X = df.drop("HeartDisease", axis=1)
y = df["HeartDisease"]

# One-hot encode categorical features
categorical_cols = X.select_dtypes(include=["object"]).columns
X_encoded = pd.get_dummies(X, columns=categorical_cols, drop_first=True)

print(f"\nFeatures after encoding: {X_encoded.shape[1]}")
print("Feature names:", list(X_encoded.columns))



Features after encoding: 15
Feature names: ['Age', 'RestingBP', 'Cholesterol', 'FastingBS', 'MaxHR', 'Oldpeak', 'Sex_M', 'ChestPainType_ATA', 'ChestPainType_NAP', 'ChestPainType_TA', 'RestingECG_Normal', 'RestingECG_ST', 'ExerciseAngina_Y', 'ST_Slope_Flat', 'ST_Slope_Up']


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_encoded, y, test_size=0.2, random_state=42, stratify=y
)

print(f"\nTrain set: {X_train.shape}, Test set: {X_test.shape}")




Train set: (734, 15), Test set: (184, 15)


In [ ]:
# 5. FEATURE SCALING (FIT ON TRAIN ONLY!)
# =============================================================================
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)  # Fit on train
X_test_scaled = scaler.transform(X_test)        # Only transform test

# Convert back to DataFrames for easier handling
X_train_scaled = pd.DataFrame(X_train_scaled, columns=X_encoded.columns)
X_test_scaled = pd.DataFrame(X_test_scaled, columns=X_encoded.columns)


In [ ]:
# =============================================================================
# 6. MODEL TRAINING & EVALUATION
# =============================================================================
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "KNN": KNeighborsClassifier(),
    "Naive Bayes": GaussianNB(),
    "Random Forest": RandomForestClassifier(random_state=42),
    "XGBoost": xgb.XGBClassifier(eval_metric="logloss", random_state=42)
}

results = {}

print("\n" + "="*50)
print("MODEL TRAINING")
print("="*50)

for name, model in models.items():
    print(f"\nTraining {name}...")
    
    # Train model
    model.fit(X_train_scaled, y_train)
    
    # Predictions
    y_pred = model.predict(X_test_scaled)
    y_proba = model.predict_proba(X_test_scaled)[:, 1]
    
    # Cross-validation score (on training set)
    cv_scores = cross_val_score(model, X_train_scaled, y_train, 
                                cv=5, scoring='roc_auc')
    
    results[name] = {
        "Accuracy": accuracy_score(y_test, y_pred),
        "ROC_AUC": roc_auc_score(y_test, y_proba),
        "Precision": precision_score(y_test, y_pred),
        "Recall": recall_score(y_test, y_pred),
        "F1 Score": f1_score(y_test, y_pred),
        "MCC": matthews_corrcoef(y_test, y_pred),
        "CV_ROC_AUC_Mean": cv_scores.mean(),
        "CV_ROC_AUC_Std": cv_scores.std()
    }


MODEL TRAINING

Training Logistic Regression...

Training Decision Tree...

Training KNN...

Training Naive Bayes...

Training Random Forest...

Training XGBoost...


In [ ]:
# =============================================================================
# 7. RESULTS SUMMARY
# =============================================================================
results_df = pd.DataFrame(results).T
results_df = results_df.round(4)

print("\n" + "="*50)
print("RESULTS SUMMARY")
print("="*50)
print(results_df)

# Find best model by ROC-AUC
best_model_name = results_df['ROC_AUC'].idxmax()
print(f"\n🏆 Best Model: {best_model_name}")
print(f"   Test ROC-AUC: {results_df.loc[best_model_name, 'ROC_AUC']:.4f}")


RESULTS SUMMARY
                     Accuracy  ROC_AUC  Precision  Recall  F1 Score     MCC  \
Logistic Regression    0.8859   0.9297     0.8716  0.9314    0.9005  0.7694   
Decision Tree          0.7880   0.7813     0.7890  0.8431    0.8152  0.5691   
KNN                    0.8859   0.9360     0.8857  0.9118    0.8986  0.7686   
Naive Bayes            0.9130   0.9451     0.9300  0.9118    0.9208  0.8246   
Random Forest          0.8696   0.9314     0.8750  0.8922    0.8835  0.7356   
XGBoost                0.8587   0.9219     0.8725  0.8725    0.8725  0.7140   

                     CV_ROC_AUC_Mean  CV_ROC_AUC_Std  
Logistic Regression           0.9211          0.0329  
Decision Tree                 0.7858          0.0275  
KNN                           0.9064          0.0293  
Naive Bayes                   0.9177          0.0347  
Random Forest                 0.9228          0.0250  
XGBoost                       0.9201          0.0157  

🏆 Best Model: Naive Bayes
   Test ROC-AUC: 

In [ ]:
# =============================================================================
# 8. SAVE MODELS
# =============================================================================
print("\n" + "="*50)
print("SAVING MODELS")
print("="*50)

for name, model in models.items():
    filename = name.replace(" ", "_") + ".pkl"
    with open(filename, "wb") as f:
        pickle.dump(model, f)
    print(f"✓ Saved {filename}")

# Save scaler with feature names
scaler.feature_names_in_ = X_encoded.columns.to_numpy()
with open("scaler.pkl", "wb") as f:
    pickle.dump(scaler, f)
print("✓ Saved scaler.pkl")

# Also save feature names separately for reference
with open("feature_names.pkl", "wb") as f:
    pickle.dump(list(X_encoded.columns), f)
print("✓ Saved feature_names.pkl")

print("\n✅ All done! Models trained and saved successfully.")


SAVING MODELS
✓ Saved Logistic_Regression.pkl
✓ Saved Decision_Tree.pkl
✓ Saved KNN.pkl
✓ Saved Naive_Bayes.pkl
✓ Saved Random_Forest.pkl
✓ Saved XGBoost.pkl
✓ Saved scaler.pkl
✓ Saved feature_names.pkl

✅ All done! Models trained and saved successfully.
